In [ ]:
import os
import xarray as xr
import pandas as pd
import numpy as np
import psutil
import gc
from tqdm import tqdm

# ============================================================================
# 1. Rutas (cámbialas según tu sistema)
# ============================================================================
ruta_precip = '/home/santiago/Escritorio/UNIVERSIDAD/FISICA_DEL_CLIMA/F-sica-del-Clima/proyecto_4/DATOS/Precipitacion_SA.nc'
ruta_nino   = '/home/santiago/Escritorio/UNIVERSIDAD/FISICA_DEL_CLIMA/F-sica-del-Clima/proyecto_4/DATOS/SOI_ANOMALY.xlsx'
carpeta_salida = '/home/santiago/Escritorio/UNIVERSIDAD/FISICA_DEL_CLIMA/F-sica-del-Clima/proyecto_4/DATOS'
os.makedirs(carpeta_salida, exist_ok=True)

# ============================================================================
# FUNCIÓN PARA MONITOREAR MEMORIA
# ============================================================================
def print_memory_usage(stage):
    mem = psutil.Process().memory_info().rss / 1024 / 1024
    print(f"[MEMORIA] {stage}: {mem:.1f} MB")

# ============================================================================
# 2. CARGAR PRECIPITACIÓN, REDUCIR RESOLUCIÓN Y RECORTAR A 1950-2018
# ============================================================================
print("Cargando precipitación...")
ds = xr.open_dataset(ruta_precip)
ds = ds.rename({'valid_time': 'time', 'latitude': 'lat', 'longitude': 'lon'})
precip = ds['tp'] * 1000   # mm/mes
print_memory_usage("Después de cargar precipitación original")

# --- REDUCCIÓN ESPACIAL (AJUSTA EL FACTOR SEGÚN TU MEMORIA) ---
coarsen_factor = 1   # 2 = 149×148; 3 = ~99×99; 4 = 74×74
precip_coarse = precip.coarsen(lat=coarsen_factor, lon=coarsen_factor, boundary='trim').mean()
print(f"Nueva dimensión espacial: {precip_coarse.lat.size} lat × {precip_coarse.lon.size} lon")
print_memory_usage("Después de coarsen")

# Asegurar que la coordenada temporal esté al inicio del mes (sin parte horaria)
precip_coarse['time'] = precip_coarse.time.dt.floor('D')

# Recortar a enero 1950 - diciembre 2018
fecha_inicio = '1951-01-01'
fecha_fin    = '2018-12-31'
precip_coarse = precip_coarse.sel(time=slice(fecha_inicio, fecha_fin))
print(f"Rango temporal precipitación: {precip_coarse.time.min().values} → {precip_coarse.time.max().values}")
print_memory_usage("Después de recorte temporal 1950-2018")

# ============================================================================
# 3. ANOMALÍA ESTANDARIZADA MENSUAL DE PRECIPITACIÓN
# ============================================================================
print("Calculando anomalía estandarizada mensual (1950-2018)...")
# Agrupar por mes y calcular media y std a lo largo de los años
monthly_mean = precip_coarse.groupby('time.month').mean('time')
monthly_std  = precip_coarse.groupby('time.month').std('time')

# Función para estandarizar cada mes (vectorizada)
def standardize_by_month(da, mean_da, std_da):
    meses = da['time'].dt.month
    std_anom = (da - mean_da.sel(month=meses)) / std_da.sel(month=meses)
    return std_anom

precip_anom_std = standardize_by_month(precip_coarse, monthly_mean, monthly_std)
print_memory_usage("Después de cálculo de anomalía estandarizada")

# ============================================================================
# 4. CARGAR ÍNDICE ENSO 3.4 Y RECORTAR AL MISMO PERÍODO
# ============================================================================
print("Cargando ENSO...")
df = pd.read_excel(ruta_nino)

# Verificar columnas
if 'YR' not in df.columns or 'MON' not in df.columns:
    raise ValueError("El archivo Excel debe tener columnas 'YR' y 'MON'")

# Crear fechas correctamente: renombrar a 'year', 'month' y añadir 'day'
df_fechas = df[['YR', 'MON']].rename(columns={'YR': 'year', 'MON': 'month'})
df_fechas['day'] = 1
df['date'] = pd.to_datetime(df_fechas)

# Seleccionar la columna (ANOM3.4 o NINO3.4)
enso = df.set_index('date')['ANOM']
enso.index = enso.index.rename('time')
enso_da = xr.DataArray(enso, dims=['time'])

# Recortar ENSO al mismo período 1950-2018
enso_da = enso_da.sel(time=slice(fecha_inicio, fecha_fin))
print(f"Rango temporal ENSO: {enso_da.time.min().values} → {enso_da.time.max().values}")
print_memory_usage("Después de cargar y recortar ENSO")

# ============================================================================
# 5. ALINEAR LAS DOS SERIES (intersección común)
# ============================================================================
# Extraer valores de tiempo como arrays de numpy
time_precip = precip_anom_std.time.values
time_enso   = enso_da.time.values

# Calcular intersección
time_comun = np.intersect1d(time_precip, time_enso)
time_comun = pd.DatetimeIndex(time_comun)  # convertir a índice para usar en sel

if len(time_comun) != len(time_precip) or len(time_comun) != len(time_enso):
    print("Advertencia: las fechas no coinciden perfectamente. Usando intersección.")
    precip_anom_std = precip_anom_std.sel(time=time_comun)
    enso_da = enso_da.sel(time=time_comun)
else:
    print("Las fechas coinciden exactamente.")

print(f"Período común para correlación: {time_comun.min()} → {time_comun.max()}")
print_memory_usage("Después de alinear precipitación y ENSO")

# ============================================================================
# 6. CORRELACIÓN CELDA POR CELDA
# ============================================================================
print("Calculando correlación celda a celda...")
latitudes = precip_anom_std.lat.values
longitudes = precip_anom_std.lon.values
corr_matrix = np.full((len(latitudes), len(longitudes)), np.nan, dtype=np.float32)

enso_values = enso_da.values  # serie ENSO (vector)

for i, lat in enumerate(tqdm(latitudes, desc="Latitudes")):
    for j, lon in enumerate(longitudes):
        prec_series = precip_anom_std.isel(lat=i, lon=j).values
        # Eliminar posibles NaNs (pueden aparecer si la desviación estándar fue cero)
        mask = ~np.isnan(prec_series)
        if np.sum(mask) > 10:   # mínimo de datos válidos
            corr_matrix[i, j] = np.corrcoef(prec_series[mask], enso_values[mask])[0, 1]
    if i % 20 == 0:
        gc.collect()
# Después de calcular corr_matrix
max_corr = np.nanmax(corr_matrix)
min_corr = np.nanmin(corr_matrix)
print(f"Correlación: mínimo = {min_corr:.4f}, máximo = {max_corr:.4f}")
print(f"Rango completo: {min_corr:.3f} a {max_corr:.3f}")
print_memory_usage("Después de correlación")

# ============================================================================
# 7. CREAR DATASET CON COORDENADAS ORDENADAS Y GUARDAR COMO NETCDF CF-COMPLIANT
# ============================================================================
print("Preparando y guardando resultado en NetCDF...")

# Crear DataArray con la matriz de correlación
corr_da = xr.DataArray(
    corr_matrix,
    dims=('lat', 'lon'),
    coords={'lat': latitudes, 'lon': longitudes},
    name='correlacion',
    attrs={
        'long_name': 'Correlación de Pearson entre anomalía estandarizada de precipitación e índice ENSO 3.4',
        'units': 'adimensional'
    }
)

# Reordenar latitud a creciente (de menor a mayor) y longitud creciente
corr_da = corr_da.sortby('lat')   # lat ahora de -58.875 a 15.125 (si original era decreciente)
corr_da = corr_da.sortby('lon')   # lon ya es creciente, pero por seguridad

# Asignar atributos estándar para que GrADS reconozca los ejes
corr_da.lat.attrs = {'standard_name': 'latitude', 'units': 'degrees_north'}
corr_da.lon.attrs = {'standard_name': 'longitude', 'units': 'degrees_east'}

# Crear Dataset con metadatos adicionales
ds_out = xr.Dataset({'correlacion': corr_da})
ds_out.attrs['title'] = 'Correlación ENSO-Precipitación en Sudamérica'
ds_out.attrs['institution'] = 'Usuario'
ds_out.attrs['source_precip'] = 'ERA5'
ds_out.attrs['source_enso'] = 'ANOM3.4 (archivo Excel)'
ds_out.attrs['period'] = f'{fecha_inicio} a {fecha_fin}'
ds_out.attrs['method'] = 'Anomalía mensual estandarizada (media y std por mes)'

# Encontrar el índice del valor máximo y mínimo (ignorando NaNs)
max_idx = np.nanargmax(corr_matrix)
min_idx = np.nanargmin(corr_matrix)
max_pos = np.unravel_index(max_idx, corr_matrix.shape)  # (i, j)
min_pos = np.unravel_index(min_idx, corr_matrix.shape)
lat_max = latitudes[max_pos[0]]
lon_max = longitudes[max_pos[1]]
lat_min = latitudes[min_pos[0]]
lon_min = longitudes[min_pos[1]]
max_val = np.nanmax(corr_matrix)
min_val = np.nanmin(corr_matrix)

print(f"Correlación máxima: {max_val:.4f} en (lat={lat_max:.3f}°, lon={lon_max:.3f}°)")
print(f"Correlación mínima: {min_val:.4f} en (lat={lat_min:.3f}°, lon={lon_min:.3f}°)")
# Valor más cercano a cero (en valor absoluto)
abs_corr = np.abs(corr_matrix)
flat_abs = abs_corr.flatten()
# Ignorar NaNs
mask_finite = ~np.isnan(flat_abs)
flat_abs_finite = flat_abs[mask_finite]
min_abs_val = np.min(flat_abs_finite)
# Índice del mínimo absoluto en la matriz aplanada
idx_zero_abs = np.nanargmin(abs_corr)   # funciona con numpy 1.17+
pos_zero = np.unravel_index(idx_zero_abs, corr_matrix.shape)
lat_zero = latitudes[pos_zero[0]]
lon_zero = longitudes[pos_zero[1]]
# El valor real (con signo)
zero_val = corr_matrix[pos_zero]

print(f"\n>>> Correlación más cercana a cero: {zero_val:.6f} (abs={min_abs_val:.6f})")
print(f"    en (lat={lat_zero:.3f}°, lon={lon_zero:.3f}°)")

ruta_nc = os.path.join(carpeta_salida, 'correlacion_SOI_precip_anomalias_cf.nc')
ds_out.to_netcdf(ruta_nc, mode='w')
print(f"Correlación guardada en: {ruta_nc}")
print_memory_usage("Final")

Cargando precipitación...
[MEMORIA] Después de cargar precipitación original: 1474.3 MB
Nueva dimensión espacial: 298 lat × 296 lon
[MEMORIA] Después de coarsen: 1466.3 MB
Rango temporal precipitación: 1951-01-01T00:00:00.000000000 → 2018-12-01T00:00:00.000000000
[MEMORIA] Después de recorte temporal 1950-2018: 1466.5 MB
Calculando anomalía estandarizada mensual (1950-2018)...
[MEMORIA] Después de cálculo de anomalía estandarizada: 1701.0 MB
Cargando ENSO...
Rango temporal ENSO: 1951-01-01T00:00:00.000000 → 2018-12-01T00:00:00.000000
[MEMORIA] Después de cargar y recortar ENSO: 1701.9 MB
Las fechas coinciden exactamente.
Período común para correlación: 1951-01-01 00:00:00 → 2018-12-01 00:00:00
[MEMORIA] Después de alinear precipitación y ENSO: 1701.9 MB
Calculando correlación celda a celda...


Latitudes: 100%|██████████| 298/298 [00:39<00:00,  7.51it/s]

Correlación: mínimo = -0.3713, máximo = 0.4095
Rango completo: -0.371 a 0.409
[MEMORIA] Después de correlación: 1430.6 MB
Preparando y guardando resultado en NetCDF...
Correlación máxima: 0.4095 en (lat=8.750°, lon=-71.250°)
Correlación mínima: -0.3713 en (lat=-14.750°, lon=-77.000°)

>>> Correlación más cercana a cero: -0.000001 (abs=0.000001)
    en (lat=-57.500°, lon=-67.000°)
Correlación guardada en: /home/santiago/Escritorio/UNIVERSIDAD/FISICA_DEL_CLIMA/F-sica-del-Clima/proyecto_4/DATOS/correlacion_SOI_precip_anomalias_cf.nc
[MEMORIA] Final: 1430.6 MB
